# Trapezoidal Rule

Divide the interval $[a, b]$ into $N$ slices with width $h = (b-a)/N$:
$$
    I(a, b) \approx \boxed{h \left[\frac{1}{2} f(a) + \frac{1}{2} f(b) + \sum^{N-1}_{k=1} f(a+kh) \right]}.
$$

In [18]:
import numpy as np
from scipy.integrate import trapezoid

def f(x):
    """f(x) = x^2"""
    return x ** 2

def I(a, b, N, h, f):
    k = np.arange(1, N)
    return h * (
        0.5 * f(a)+ 0.5 * f(b)
        + np.sum(f(a + k * h))
    )

def exact(a, b):
    return (1/3) * (b ** 3 - a ** 3)

N = 10
a, b = 0, 5
h = (b - a) / N    

x = np.linspace(a, b, N + 1)

result = I(a, b, N, h, f)
exact_result = exact(a, b)

print(f'Result: {result}')
print(f'Exact: {exact_result:.3f}')
print(f'SciPy: {trapezoid(f(x), x)}')

Result: 41.875
Exact: 41.667
SciPy: 41.875


# Simpson's Rule

Fit a quadratic under every set of 3 points and sum their areas:
$$
    I(a, b) \approx \boxed{\frac{1}{3} h \left[f(a) + f(b) + 4 \sum_{\substack{k \, \text{odd} \\ {1 \ldots N-1}}} f(a+kh) + 2 \sum_{\substack{k \text{ even} \\ 2 \dots N-2}} f(a+kh)  \right]}. 
$$

In [25]:
import numpy as np
from scipy.integrate import simpson

def f(x):
    """f(x) = x^2"""
    return x ** 2

def I(a, b, N, h, f):

    k_odd = np.arange(1, N, 2)
    k_even = np.arange(2, N - 1, 2)

    sum_odd = np.sum(f(a + k_odd * h))
    sum_even = np.sum(f(a + k_even * h))

    return (1/3) * h * (
        f(a) + f(b) + 4 * sum_odd + 2 * sum_even
    )

def exact(a, b):
    return (1/3) * (b ** 3 - a ** 3)

def scipy_sol(x):
    return simpson(f(x), x=x)

a, b = 0, 5
N = 8
h = (b - a) / N
x = np.linspace(a, b, N + 1)

print('--- Results --- \n')
print(f'Simpson: {I(a, b, N, h, f)}')
print(f'Exact: {exact(a, b)}')
print(f'SciPy: {scipy_sol(x)}')


--- Results --- 

Simpson: 41.666666666666664
Exact: 41.666666666666664
SciPy: 41.66666666666667


# Errors

## Theoretical Approximation Errors

Use when $f(x)$ and its derivatives are simple at $a$ and $b$.

### Trapezoidal Rule Error

$$
    \epsilon = \boxed{\frac{1}{12} h^2 [f'(a) - f'(b)]}.
$$
This tells us how much area is missing.

### Simpson's Rule Error 

$$
    \epsilon = \boxed{\frac{1}{90} h^4 \left[ f'''(a) - f'''(b) \right]}.
$$

In [27]:
import numpy as np

def f(x):
    return x ** 4

def f_first(x):
    return 4 * (x ** 3)

def f_third(x):
    return 24 * x

a, b = 0, 1
N = 10
h = (b - a) / N
x = np.linspace(a, b, N + 1)

error_trap = (1/12) * (h ** 2) * (f_first(a) - f_first(b))
error_simps = (1/90) * (h ** 4) * (f_third(a) - f_third(b))

print(f'Error Trapezoidal = {error_trap}')
print(f'Error Simpson = {error_simps}')

Error Trapezoidal = -0.003333333333333334
Error Simpson = -2.6666666666666673e-05


## Practical Error Estimation

When we don't have a simple function. Integrate twice, first with $N$ slices to get $I_1$ and then with $2N$ slices to get $I_2$.

### Trapezoidal Practical Error

$$
    \epsilon_2 = \boxed{\frac{1}{3}(I_2 - I_1)}.
$$


### Simpson's Practical Error

$$
    \epsilon_2 = \boxed{\frac{1}{15}(I_2 - I_1)}.
$$

In [31]:
import numpy as np

def f(x):
    return x ** 4

def trapezoid(a, b, N, h, f):
    k = np.arange(1, N)
    return h * (
        0.5 * f(a)+ 0.5 * f(b)
        + np.sum(f(a + k * h))
    )

def simpson(a, b, N, h, f):

    k_odd = np.arange(1, N, 2)
    k_even = np.arange(2, N - 1, 2)

    sum_odd = np.sum(f(a + k_odd * h))
    sum_even = np.sum(f(a + k_even * h))

    return (1/3) * h * (
        f(a) + f(b) + 4 * sum_odd + 2 * sum_even
    )

a, b = 0, 1
N = 10
h = (b - a) / N
x = np.linspace(a, b, N + 1)

I1_trap = trapezoid(a, b, N, h, f)
I2_trap = trapezoid(a, b, 2*N, h, f)

I1_simp = simpson(a, b, N, h, f)
I2_simp = simpson(a, b, 2*N, h, f)

error2_trap = (1/3) * (I2_trap - I1_trap)
error2_simp = (1/15) * (I2_simp - I1_simp)

print(f'Trapezoidal Practical Error: {error2_trap:.2f}')
print(f'Simpson Practical Error: {error2_simp:.2f}')


Trapezoidal Practical Error: 1.82
Simpson Practical Error: 0.38


## Optimal Number of Slices


### Trapezoidal Rule Optimal
Note that $C$ is the rounding error constant of the computer, $\sim10^{-16}$ in Python.

$$
    N \approx \boxed{(b - a) \sqrt{ \frac{f'(a) - f'(b)}{12 \int_a^b f(x) \, dx} } C^{-1/2}}.
$$


### Trapezoidal Rule Optimal

$$
    N = \boxed{(b - a)^4 \sqrt[4]{ \frac{f'''(a) - f'''(b)}{90 \int_a^b f(x) \, dx} } C^{-1/4}}.
$$

In [43]:
import numpy as np

def f(x):
    return x ** 4

def f_first(x):
    return 4 * (x ** 3)

def f_third(x):
    return 24 * x

def trapezoid(a, b, N, h, f):
    k = np.arange(1, N)
    return h * (
        0.5 * f(a)+ 0.5 * f(b)
        + np.sum(f(a + k * h))
    )

def simpson(a, b, N, h, f):

    k_odd = np.arange(1, N, 2)
    k_even = np.arange(2, N - 1, 2)

    sum_odd = np.sum(f(a + k_odd * h))
    sum_even = np.sum(f(a + k_even * h))

    return (1/3) * h * (
        f(a) + f(b) + 4 * sum_odd + 2 * sum_even
    )

a, b = 0, 1
N = 10
h = (b - a) / N
x = np.linspace(a, b, N + 1)

C = np.finfo(np.float64).eps

N_trap = (b - a) * np.sqrt(
    np.abs((f_first(a) - f_third(b)) / 12 * trapezoid(a, b, N, h, f)
) * (C ** (-0.5)))

N_simp = (((b - a) ** 4) * (
    np.abs((f_third(a) - f_third(b)) / 90 * simpson(a, b, N, h, f)
) ** (1/4)) * (C ** (-0.25)))

print(f'{N_trap:.0f}')
print(f'{N_simp:.0f}')

5224
3937
